<a href="https://colab.research.google.com/github/s32230019/UAS-PENGOLAHAN-CITRA-DIGITAL/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UAS PENGOLAHAN CITRA DIGITAL

In [ ]:
# 1. IMPORT LIBRARY YANG DIBUTUHKAN
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from google.colab import files
import io
from PIL import Image

# Variabel Global untuk menampung data gambar
img_rgb_global = None
gambar_hasil_akhir = None

print("Library berhasil di-import!")

In [ ]:
# 2. FUNGSI UNTUK MENGUNGGAH GAMBAR KE COLAB
def upload_gambar_colab():
    global img_rgb_global
    print("Silakan pilih dan unggah gambar...")
    uploaded = files.upload()

    if not uploaded:
        print("Upload dibatalkan.")
        return

    # Mengambil file pertama yang diunggah
    file_name = list(uploaded.keys())[0]
    file_data = uploaded[file_name]

    # Membaca gambar menggunakan OpenCV
    nparr = np.frombuffer(file_data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if img is None:
        print("Gagal membaca gambar! Pastikan formatnya benar (JPG/PNG).")
        return

    # Konversi dari BGR ke RGB
    img_rgb_global = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize jika gambar terlalu besar agar responsif
    max_w, max_h = 850, 360
    h, w = img_rgb_global.shape[:2]
    if w > max_w or h > max_h:
        rasio = min(max_w / w, max_h / h)
        img_rgb_global = cv2.resize(img_rgb_global, (int(w * rasio), int(h * rasio)), interpolation=cv2.INTER_AREA)

    print(f"Gambar '{file_name}' berhasil dimuat!")

# Langsung panggil fungsinya begitu cell ini dijalankan
upload_gambar_colab()

In [ ]:
# 3. FUNGSI PROSES DAN VISUALISASI GAMBAR
def proses_dan_tampilkan(gamma, clahe_val, denoise, sharpen, saturation):
    global img_rgb_global, gambar_hasil_akhir
    if img_rgb_global is None:
        print("Belum ada gambar yang diunggah. Jalankan ulang Cell 2.")
        return

    temp = img_rgb_global.copy()

    # A. Proses Denoise (Pembersih Bintik)
    if denoise > 0:
        temp = cv2.fastNlMeansDenoisingColored(temp, None, denoise, denoise, 7, 21)

    # B. Proses Gamma Correction (Kecerahan)
    val_gamma = gamma / 100.0
    if val_gamma != 1.0:
        val_gamma = max(val_gamma, 0.1)
        temp = np.array(255 * (temp / 255) ** val_gamma, dtype='uint8')

    # C. Proses CLAHE (Kontras Lokal)
    val_clahe = clahe_val / 10.0
    if val_clahe > 0:
        lab = cv2.cvtColor(temp, cv2.COLOR_RGB2LAB)
        l, a, b_ch = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=val_clahe, tileGridSize=(8,8))
        cl = clahe.apply(l)
        merged = cv2.merge((cl, a, b_ch))
        temp = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    # D. Proses Color Boost (Saturasi)
    if saturation > 0:
        hsv = cv2.cvtColor(temp, cv2.COLOR_RGB2HSV)
        h, s, v = cv2.split(hsv)
        s = cv2.add(s, saturation)
        temp = cv2.merge((h, s, v))
        temp = cv2.cvtColor(temp, cv2.COLOR_HSV2RGB)

    # E. Proses Sharpen (Ketajaman)
    if sharpen > 0:
        amt = sharpen / 10.0
        kernel = np.array([
            [0, -amt, 0],
            [-amt, 1 + (4 * amt), -amt],
            [0, -amt, 0]
        ])
        temp = cv2.filter2D(temp, -1, kernel)

    gambar_hasil_akhir = temp

    # Menampilkan Perbandingan Gambar Asli vs Hasil Pemrosesan
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(img_rgb_global)
    axes[0].set_title("GAMBAR ASLI", fontsize=14, fontweight='bold', color='gray')
    axes[0].axis('off')

    axes[1].imshow(temp)
    axes[1].set_title("HASIL PEMROSESAN", fontsize=14, fontweight='bold', color='orange')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# 4. MEMBUAT TAMPILAN INTERAKTIF WIDGET (SLIDER DAN BUTTON LENGKAP)

# Inisialisasi Slider Komponen
w_gamma = widgets.IntSlider(min=10, max=200, step=1, value=100, description='Gamma:')
w_clahe = widgets.IntSlider(min=0, max=50, step=1, value=0, description='CLAHE:')
w_denoise = widgets.IntSlider(min=0, max=20, step=1, value=0, description='Denoise:')
w_sharpen = widgets.IntSlider(min=0, max=30, step=1, value=0, description='Sharpen:')
w_saturation = widgets.IntSlider(min=0, max=100, step=1, value=0, description='Color Boost:')

# Fungsi Logika untuk tombol Auto Enhance
def auto_enhance_trigger(b):
    global img_rgb_global
    if img_rgb_global is None:
        return
    gray = cv2.cvtColor(img_rgb_global, cv2.COLOR_RGB2GRAY)
    rata_kecerahan = np.mean(gray)

    if rata_kecerahan < 40:
        w_gamma.value = 65; w_clahe.value = 15; w_sharpen.value = 8; w_saturation.value = 15
    elif rata_kecerahan < 90:
        w_gamma.value = 75; w_clahe.value = 12; w_sharpen.value = 6; w_saturation.value = 10
    else:
        w_gamma.value = 90; w_clahe.value = 8; w_sharpen.value = 10; w_saturation.value = 5
    print("Auto Enhance Berhasil Diterapkan!")

# Fungsi Logika untuk tombol Reset Parameter
def reset_sliders_trigger(b):
    w_gamma.value = 100
    w_clahe.value = 0
    w_denoise.value = 0
    w_sharpen.value = 0
    w_saturation.value = 0
    print("Semua pengaturan berhasil direset!")

# Fungsi Logika untuk mendownload gambar
def download_gambar(b):
    global gambar_hasil_akhir
    if gambar_hasil_akhir is not None:
        save_img = cv2.cvtColor(gambar_hasil_akhir, cv2.COLOR_RGB2BGR)
        cv2.imwrite('hasil_pemrosesan.jpg', save_img)
        files.download('hasil_pemrosesan.jpg')
        print("Gambar berhasil didownload!")

# Menyusun Kontrol Output Interaktif
out = widgets.interactive_output(proses_dan_tampilkan, {
    'gamma': w_gamma, 'clahe_val': w_clahe, 'denoise': w_denoise, 'sharpen': w_sharpen, 'saturation': w_saturation
})

# Membuat Komponen Button (Auto Enhance, Reset, dan Simpan)
btn_auto = widgets.Button(description="Auto Enhance", button_style='warning')
btn_auto.on_click(auto_enhance_trigger)

btn_reset = widgets.Button(description="Reset Parameter", button_style='info')
btn_reset.on_click(reset_sliders_trigger)

btn_download = widgets.Button(description="Simpan Gambar", button_style='success')
btn_download.on_click(download_gambar)

# Merangkai Layout Baris Tombol agar berjejer rapi
box_sliders = widgets.VBox([w_gamma, w_clahe, w_denoise, w_sharpen, w_saturation])
box_buttons = widgets.HBox([btn_auto, btn_reset, btn_download])
ui = widgets.VBox([box_sliders, box_buttons])

# Tampilkan UI Aplikasi di bawah cell
display(ui, out)